# 18 — Multi-timeframe trendline trading simulation

Trades fitted support/resistance lines from several timeframes at once. **Long** breaks *up* through
a resistance line, **short** breaks *down* through a support line. Stop = the previous `STOP_N`-bar
extreme; exit = breakeven at 1R, target at 3R.

Edit the config below, run all, look at the chart. Everything else is in the appendix.

> Trendline breaks have already been measured here and carry **no directional edge** — the hit rate
> cleared 0.50 in 36% of 102 cells, *below* chance ([`docs/WHAT_WORKS.md`](../docs/WHAT_WORKS.md)).
> This notebook shows you what a rule did; it is not evidence that it works.


## Config

In [ ]:
import sys, os, warnings; warnings.filterwarnings('ignore')
try:
    import quant
except ModuleNotFoundError:
    sys.path.insert(0, os.path.abspath('..')); import quant
import numpy as np, pandas as pd

SYMBOL, SOURCE = "BTCUSDT", "binance"
TF          = "5m"                       # chart + execution timeframe
LINE_TFS    = ("15m", "1h")              # ALSO trade lines fitted on these
VIEW_START, END = "2026-06-01", "2026-06-10"
WARMUP_START    = "2026-05-01"           # data starts here so the lines have pivots
TF_WARMUP_BARS  = 800                    # history per extra timeframe, in ITS OWN bars

# HOW MANY CANDLES TO WAIT before taking a break. Per timeframe, because two 15m closes is half an
# hour and two 1m closes is two minutes. From experiments/trendline_wait_search.py over 5.5 months
# in 6 windows — read the appendix for how much of it to believe (short answer: the 5m number).
TL_CLOSE_N  = {"5m": 4, "15m": 5, "1h": 4, "*": 2}   # or just an int for every timeframe
TL_FULL_M   = 0          # consecutive candles ENTIRELY beyond the line (0 = off; also per-tf)
TL_TOUCH_K  = 0          # the line must have had this many touches (0 = off; also per-tf)
TL_COMBINE  = "all"      # "all" = every enabled condition, "any" = one of them
TL_TOL_ATR  = 0.0        # how far past the line counts as past it, in ATR

TRADE_LONG, TRADE_SHORT = True, True

STOP_N, STOP_BUFFER_PCT = 20, 0.05       # stop at the last N bars' extreme, pushed out this far
BE_AT_RR, TP_RR         = 1.0, 3.0       # breakeven at 1R, target at 3R

COSTS = dict(initial_cash=10_000, fee_bps=8, slippage_bps=1.5,
             sizing_mode="risk_pct_equity", sizing_value=1.0,
             exit_enabled=False, max_open_trades=1)

TRENDLINE = dict(atr_mult=3.5, min_touches=3, lookback_pivots=16, line_tol_atr=0.6,
                 min_span_bars=40, max_age_bars=500, break_buffer_atr=0.0)

HEIGHT, MAX_PICKER, MAX_SHAPES_TF, MAX_TRADE_PICKER = 900, 24, 12, 40
SHOW_SR_ZONES, ZONE_LOOKBACK = True, 2000

## Run

In [ ]:
from quant.data import get_ohlcv, tf_frames
from quant.indicators.trendlines import mtf_trendlines
from quant.scenarios import run_pieces
from quant.scenarios.entries import trendline_trigger
from quant.strategies.conditions import raw

df = get_ohlcv(SYMBOL, TF, start=WARMUP_START, end=END, source=SOURCE, tz="UTC").reset_index(drop=True)
FRAMES = tf_frames(SYMBOL, LINE_TFS, view_start=VIEW_START, end=END,
                   warmup_bars=TF_WARMUP_BARS, source=SOURCE) if LINE_TFS else {}
D = mtf_trendlines(df, FRAMES, **TRENDLINE)

TF_NAMES = {"": TF, **{f"{k}__": k for k in FRAMES}}
PARAMS = dict(tl_prefixes=tuple(TF_NAMES), tl_tf_names=TF_NAMES,
              tl_close_n=TL_CLOSE_N, tl_full_m=TL_FULL_M, tl_touch_k=TL_TOUCH_K,
              tl_combine=TL_COMBINE, tl_tol_atr=TL_TOL_ATR,
              stop_n=STOP_N, stop_buffer_pct=STOP_BUFFER_PCT, be_at_rr=BE_AT_RR, tp_rr=TP_RR)

# the warm-up feeds the lines and the stop lookback; it is not traded
_vs = pd.Timestamp(VIEW_START, tz="UTC")
IN_WINDOW = raw(lambda d: (pd.to_datetime(d["t"]) >= _vs).to_numpy(), "in_window")

sides = [s for s, on in (("long", TRADE_LONG), ("short", TRADE_SHORT)) if on]
RUNS = {s: run_pieces(D, f"mtf_trendline_{s}@v1", "high_of_n@v1", "be_ladder@v1",
                      params=PARAMS, cfg=COSTS, entry_filter=IN_WINDOW) for s in sides}
TRADES = pd.concat([trendline_trigger(r.df, r.trades, PARAMS, prefix_names=TF_NAMES)
                    for r in RUNS.values()], ignore_index=True) \
           .sort_values("entry_time").reset_index(drop=True)

_risk = (TRADES.entry_price - TRADES.stop_price).abs()
TRADES["R"] = TRADES.pnl / (_risk * TRADES.qty)
TRADES["risk_%"] = 100 * _risk / TRADES.entry_price
TRADES["tid"] = [f"#{i+1:02d}" for i in range(len(TRADES))]
assert not len(TRADES) or TRADES.entry_time.min() >= _vs

print(f"{len(df):,} {TF} bars | " + " ".join(f"{k} {len(v):,}" for k, v in FRAMES.items())
      + f"\n{len(TRADES)} trades   {TRADES.R.sum():+.1f}R   "
        f"{100*(TRADES.pnl>0).mean():.0f}% win   {100*TRADES.pnl.sum()/COSTS['initial_cash']:+.2f}%")

## Chart

| drawn | |
|---|---|
| ▲ ▼ | entry — hover for R, close reason and the line it broke |
| **grey** | the entry level |
| **red dotted** | the stop, **as set at entry** |
| **green dotted** | the 3R target |
| ✕ | the exit, green if it made money |

Five dropdowns: timeframe → pattern → occurrence, then **longs/shorts/winners/losers → one trade**.
Picking a trade clears the chart to that trade and the line that caused it.

A row tagged **`sl@be`** was stopped out on a stop that had already walked to breakeven — so its
exit sits at the entry level, nowhere near the red line, and the loss is only the fees.

In [ ]:
from quant.viz import pattern_figure

fig = pattern_figure(
    D, patterns=[], trendlines=True, trendline_kwargs=TRENDLINE, tf_frames=FRAMES,
    view_start=VIEW_START, emas=(), chart_tf=TF,
    sr_zones=SHOW_SR_ZONES, zone_lookback_bars=ZONE_LOOKBACK,
    max_picker_instances=MAX_PICKER, max_shapes_per_tf=MAX_SHAPES_TF,
    max_trade_picker=MAX_TRADE_PICKER,
    trades=TRADES, trade_tp_rr=TP_RR, start_visible=("trades", "sr_zones"),
    title=f"{SYMBOL} {TF}  {VIEW_START} to {END}  ·  "
          f"{'/'.join((TF,) + tuple(FRAMES))} trendline breaks  ·  {len(TRADES)} trades",
    height=HEIGHT)
fig

## Results

In [ ]:
by_side = TRADES.groupby("side").agg(
    trades=("R", "size"), win=("pnl", lambda z: f"{100*(z>0).mean():.0f}%"),
    total_R=("R", "sum"), avg_R=("R", "mean"))
by_why = TRADES.groupby("close_reason").agg(n=("R", "size"), total_R=("R", "sum"))
by_line = TRADES.groupby("trigger").agg(n=("R", "size"), total_R=("R", "sum"),
                                        win=("pnl", lambda z: (z > 0).mean()))
for name, t in (("by side", by_side), ("why it ended", by_why), ("by timeframe", by_line)):
    print(f"\n{name}\n{t.round(2).to_string()}")

cols = ["tid", "entry_time", "side", "cause", "entry_price", "stop_price", "exit_price",
        "risk_%", "R", "close_reason"]
TRADES[cols].round(3)

---

## Appendix

### How many candles to wait — where `TL_CLOSE_N` came from

`experiments/trendline_wait_search.py` sweeps the wait per timeframe and side over **5.5 months in
6 windows**, scoring by **windows positive** rather than total R, because one huge trade in one
window is exactly what a single total hides. Nine days would give a handful of trades per cell, and
picking the argmax of six settings on ten trades is how this project has produced every result it
later had to retract.

| tf | side | trades at the pick | windows + | total R | believe it? |
|---|---|---|---|---|---|
| **5m** | long | 53 | 4/6 | −2.0R | **yes, partly** — R improves monotonically from wait 1 (−33.8R) to 4 |
| **5m** | short | 54 | 4/6 | +1.3R | same shape, −8.0R at wait 1 → +1.3R at 4 |
| 15m | short | 19 | 4/6 | +9.4R | thin |
| 15m | long | 35 | 3/6 | −9.2R | negative at every wait |
| 1h | either | 9–11 | 1–3/6 | — | **no** — too few trades to say anything |

**Nothing cleared 6 of 6.** Six settings over six windows produce a 6/6 winner by luck about 9% of
the time, so a best-of-4/6 is entirely ordinary. What is worth something is the **monotone trend on
5m**: waiting longer improves both total R *and* per-trade R (−0.32 → −0.08 for longs), so it is not
just "fewer trades paying fewer fees". That is why the 5m number is 4 and why the 1h number is set
to match rather than to its own noisy argmax.

The other thing the sweep says loudly: **longs lose at every wait on every timeframe** and shorts do
not. That is a downtrending sample, not an edge.

Re-run it yourself, or narrow it:

```
python experiments/trendline_wait_search.py
python experiments/trendline_wait_search.py --tf 5m --waits 1 2 3 4 5 6 --windows 8
```

### The cost floor

A tight stop means a large position for the same 1% risk, and fees scale with position size — so a
round trip can cost a large fraction of what you are risking. Run this to see it on your own trades:

```python
fee_R = TRADES.fees / ((TRADES.entry_price - TRADES.stop_price).abs() * TRADES.qty)
c = float(fee_R.median())
print(f"median risk {TRADES['risk_%'].median():.3f}% of price | cost {c:.2f}R per round trip")
print(f"break-even win rate p* = (1+{c:.2f})/(1+{TP_RR}) = {(1+c)/(1+TP_RR):.1%}"
      f"   vs actual {100*(TRADES.pnl>0).mean():.1f}%")
```

### The knobs

| | |
|---|---|
| `TL_CLOSE_N` | closes beyond the line before entering. Int, or a dict per timeframe with `"*"` as the fallback |
| `TL_FULL_M` | candles *entirely* beyond it, wick included. Much stricter |
| `TL_TOUCH_K` | only trade lines that had been respected this many times |
| `TL_COMBINE` | `"all"` intersects the enabled conditions, `"any"` unions them |
| `STOP_N` | stop lookback. **Larger = wider stop = smaller position = less cost in R** |
| `BE_AT_RR` / `TP_RR` | the exit ladder. `BE_AT_RR=0` disables the breakeven move |
| `LINE_TFS` | which timeframes contribute lines. `()` trades `TF` lines only |

### What would make this fair rather than plausible

None of it is done here: difference against a within-day-shuffled null
(`experiments/null_control.py`); beat the `time_entry` metronome through the same exits; count
survivors against `n_configs × 0.5^n_windows`; and trim the top K trades, not just the top one.
